# OCR Inference

Load an image and run OCR with one or more OCR models supported by `attack_ocr.py`.


In [ ]:
from pathlib import Path
import importlib
import os
import sys

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

import torch
from IPython.display import display
from PIL import Image
from transformers import (
    AutoModel,
    AutoProcessor,
    AutoTokenizer,
    DonutProcessor,
    NougatProcessor,
    Qwen2VLForConditionalGeneration,
    VisionEncoderDecoderModel,
)


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the repo root from the current working directory. "
        "Launch the notebook from this repository or one of its subdirectories."
    )


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import attack_ocr

attack_ocr = importlib.reload(attack_ocr)

from attack_ocr import (
    MODEL_CONFIGS as ATTACK_MODEL_CONFIGS,
    MODEL_INPUT_SIZE,
    MAX_NEW_TOKENS,
    build_deepseek_prompt_inputs,
    build_encoder_decoder_prompt_inputs,
    build_model_vision_inputs,
    ensure_deepseek_transformers_compat,
    ensure_nougat_dependencies,
    generate_deepseek_text,
    generate_encoder_decoder_text,
    is_encoder_decoder_family,
    resolve_deepseek_crop_grid,
    resolve_deepseek_layout,
    validate_deepseek_image_token_alignment,
)
from attacks.common import load_image_tensor
from attacks.prompting import build_chat_prompt_inputs, generate_greedy_text

SUPPORTED_MODEL_KEYS = ("deepseek_ocr_2", "imgscope", "donut", "nougat")
MODEL_CONFIGS = {key: ATTACK_MODEL_CONFIGS[key] for key in SUPPORTED_MODEL_KEYS}

print(f"Repo root: {REPO_ROOT}")


In [ ]:
IMG_IDX = 1
BUDGET = 12
RUN_MODE = "all"  # Options: "single", "all"
MODEL_KEY = "deepseek_ocr_2"  # Used when RUN_MODE == "single"
ADV_IMAGE_MODEL_KEY = "deepseek_ocr_2"
MAX_NEW_TOKENS_OVERRIDE = None
IMAGE_PATH = REPO_ROOT / "results" / f"{ADV_IMAGE_MODEL_KEY}_ocr_{BUDGET}_adv_{IMG_IDX}.png"

IMAGE_PATH = Path(IMAGE_PATH).expanduser()
if not IMAGE_PATH.is_absolute():
    IMAGE_PATH = (REPO_ROOT / IMAGE_PATH).resolve()

supported_keys = sorted(MODEL_CONFIGS)
if RUN_MODE not in {"single", "all"}:
    raise ValueError(f"RUN_MODE must be 'single' or 'all'. Got {RUN_MODE!r}.")

if ADV_IMAGE_MODEL_KEY not in MODEL_CONFIGS:
    supported = ", ".join(supported_keys)
    raise ValueError(f"ADV_IMAGE_MODEL_KEY must be one of: {supported}. Got {ADV_IMAGE_MODEL_KEY!r}.")

if RUN_MODE == "single" and MODEL_KEY not in MODEL_CONFIGS:
    supported = ", ".join(supported_keys)
    raise ValueError(f"MODEL_KEY must be one of: {supported}. Got {MODEL_KEY!r}.")

MODEL_KEYS_TO_RUN = [MODEL_KEY] if RUN_MODE == "single" else list(SUPPORTED_MODEL_KEYS)

if MAX_NEW_TOKENS_OVERRIDE is not None:
    if not isinstance(MAX_NEW_TOKENS_OVERRIDE, int) or MAX_NEW_TOKENS_OVERRIDE <= 0:
        raise ValueError("MAX_NEW_TOKENS_OVERRIDE must be a positive integer or None.")

MAX_NEW_TOKENS_TO_USE = MAX_NEW_TOKENS if MAX_NEW_TOKENS_OVERRIDE is None else MAX_NEW_TOKENS_OVERRIDE

print(f"Run mode: {RUN_MODE}")
print(f"Model keys to run: {MODEL_KEYS_TO_RUN}")
print(f"Adversarial image model key: {ADV_IMAGE_MODEL_KEY}")
print(f"Image path: {IMAGE_PATH}")
print(f"Supported model keys: {supported_keys}")
print(f"Max new tokens: {MAX_NEW_TOKENS_TO_USE}")


In [ ]:
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

input_image = Image.open(IMAGE_PATH).convert("RGB")
display(input_image)
print(f"Image path: {IMAGE_PATH}")
print(f"Image size: {input_image.size}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"[Info] Using CUDA device: {torch.cuda.get_device_name(0)}")
else:
    model_dtype = torch.float32
    print("[Warning] CUDA not available. OCR inference will run on CPU and may be slow.")

print(f"[Info] Torch dtype: {model_dtype}")


In [ ]:
print("[Info] Model loading is handled inside the inference loop.")
print(f"[Info] Models queued: {MODEL_KEYS_TO_RUN}")


In [ ]:
MODEL_OUTPUTS = {}

for model_idx, model_key in enumerate(MODEL_KEYS_TO_RUN, start=1):
    model_config = MODEL_CONFIGS[model_key]
    model_name = model_config["model_name"]
    model_family = model_config["model_family"]
    ocr_prompt = model_config["ocr_prompt"]

    print("=" * 80)
    print(f"[{model_idx}/{len(MODEL_KEYS_TO_RUN)}] Model key: {model_key}")
    print(f"Model: {model_name}")
    print(f"Family: {model_family}")
    print(f"Image path: {IMAGE_PATH}")

    processor = None
    tokenizer = None
    model = None
    state = None
    image_tensor = None
    prompt_model_inputs = None
    prompt_image_mask = None
    vision_inputs = None
    ocr_text = None

    try:
        image_tensor = load_image_tensor(IMAGE_PATH, device)
        print(f"[Info] Loading model: {model_name}")

        if model_family == "deepseek":
            ensure_deepseek_transformers_compat()
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModel.from_pretrained(
                model_name,
                trust_remote_code=True,
                use_safetensors=True,
                torch_dtype=model_dtype,
            ).to(device)
            model.eval()
            model.requires_grad_(False)

            state = {
                "device": device,
                "dtype": model_dtype,
                "model_family": model_family,
                "prompt": ocr_prompt,
                "base_size": model_config["base_size"],
                "image_size": model_config["image_size"],
                "crop_mode": model_config["crop_mode"],
                "deepseek_layout": resolve_deepseek_layout(model),
                "deepseek_crop_grid": resolve_deepseek_crop_grid(
                    int(image_tensor.shape[-2]),
                    int(image_tensor.shape[-1]),
                    crop_mode=model_config["crop_mode"],
                    image_size=model_config["image_size"],
                ),
            }
            if state["deepseek_layout"] != "ocr2":
                raise RuntimeError(f"Unsupported DeepSeek layout: {state['deepseek_layout']}")

            _, prompt_model_inputs, prompt_image_mask = build_deepseek_prompt_inputs(tokenizer, state)
            state["deepseek_prompt_image_mask"] = prompt_image_mask
            prompt_token_count = prompt_model_inputs["input_ids"].shape[1]
            vision_inputs = build_model_vision_inputs(state, image_tensor.squeeze(0))
            validate_deepseek_image_token_alignment(state, vision_inputs)

            with torch.no_grad():
                ocr_text = generate_deepseek_text(
                    model,
                    tokenizer,
                    prompt_model_inputs,
                    prompt_token_count,
                    vision_inputs,
                    max_new_tokens=MAX_NEW_TOKENS_TO_USE,
                )
        elif is_encoder_decoder_family(model_family):
            if model_family == "encoder_decoder_nougat":
                ensure_nougat_dependencies()
                processor = NougatProcessor.from_pretrained(model_name, backend="torchvision")
            else:
                processor = DonutProcessor.from_pretrained(model_name, backend="torchvision", use_fast=False)

            model = VisionEncoderDecoderModel.from_pretrained(
                model_name,
                torch_dtype=model_dtype,
            ).to(device)
            model.eval()
            model.requires_grad_(False)

            prompt_model_inputs = build_encoder_decoder_prompt_inputs(
                processor.tokenizer,
                ocr_prompt,
                device,
            )
            state = {
                "device": device,
                "dtype": model_dtype,
                "model_family": model_family,
                "image_processor": processor.image_processor,
                **prompt_model_inputs,
            }
            vision_inputs = build_model_vision_inputs(state, image_tensor.squeeze(0))

            with torch.no_grad():
                ocr_text = generate_encoder_decoder_text(
                    model,
                    processor,
                    state,
                    vision_inputs,
                    max_new_tokens=MAX_NEW_TOKENS_TO_USE,
                )
        elif model_family == "qwen":
            processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
            model = Qwen2VLForConditionalGeneration.from_pretrained(
                model_name,
                trust_remote_code=True,
                torch_dtype=model_dtype,
            ).to(device)
            model.eval()
            model.requires_grad_(False)

            vision_config = model.config.vision_config
            state = {
                "device": device,
                "dtype": model_dtype,
                "model_family": model_family,
                "model_input_size": MODEL_INPUT_SIZE,
                "patch_size": vision_config.patch_size,
                "temporal_patch_size": vision_config.temporal_patch_size,
                "merge_size": vision_config.spatial_merge_size,
                "mean": torch.tensor(processor.image_processor.image_mean, device=device, dtype=torch.float32),
                "std": torch.tensor(processor.image_processor.image_std, device=device, dtype=torch.float32),
            }

            _, prompt_model_inputs = build_chat_prompt_inputs(
                processor,
                device,
                ocr_prompt,
                (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE),
            )
            prompt_token_count = prompt_model_inputs["input_ids"].shape[1]
            vision_inputs = build_model_vision_inputs(state, image_tensor.squeeze(0))

            with torch.no_grad():
                ocr_text = generate_greedy_text(
                    model,
                    processor,
                    prompt_model_inputs,
                    prompt_token_count,
                    vision_inputs,
                    max_new_tokens=MAX_NEW_TOKENS_TO_USE,
                )
        else:
            raise RuntimeError(f"MODEL_KEY={model_key!r} with family {model_family!r} is not supported by this notebook.")

        print()
        print("Extracted OCR text:")
        print(ocr_text if ocr_text else "<empty output>")

        MODEL_OUTPUTS[model_key] = {
            "model_name": model_name,
            "family": model_family,
            "ocr_text": ocr_text,
        }
    except Exception as exc:
        error_text = f"{type(exc).__name__}: {exc}"
        MODEL_OUTPUTS[model_key] = {
            "model_name": model_name,
            "family": model_family,
            "error": error_text,
        }
        print(f"[Error] {error_text}")
        if RUN_MODE == "single":
            raise
        print("[Info] Continuing to next model.")
    finally:
        if vision_inputs is not None:
            del vision_inputs
        if prompt_image_mask is not None:
            del prompt_image_mask
        if prompt_model_inputs is not None:
            del prompt_model_inputs
        if state is not None:
            del state
        if image_tensor is not None:
            del image_tensor
        if model is not None:
            del model
        if processor is not None:
            del processor
        if tokenizer is not None:
            del tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()
            if hasattr(torch.cuda, "ipc_collect"):
                torch.cuda.ipc_collect()

print("=" * 80)
print(f"Completed {len(MODEL_OUTPUTS)} model run(s).")
